# Week 14: A Small FastAPI Service

Same logic as `call_api.py`. Requires `LLM_API_KEY` and `LLM_MODEL` in a `.env` file (see `.env.example`) for `/ask`, which calls a real LLM. `/health` and `/search` need no API key. Reuses the persistent collection Week 9 built (`data/processed/chroma`) — run `examples/week-09/build_passage_index.py` first if you haven't already.

This starts a real `uvicorn` server in a background thread and talks to it over a real TCP socket with `httpx` — not FastAPI's in-process `TestClient` (that's what the test suite uses).

In [ ]:
import threading
import time

import httpx
import uvicorn
from dotenv import load_dotenv

from ai_finance_course.api import app

HOST = "127.0.0.1"
PORT = 8014
BASE_URL = f"http://{HOST}:{PORT}"


def _run_server() -> None:
    uvicorn.run(app, host=HOST, port=PORT, log_level="warning")


def _wait_for_server(timeout: float = 10.0) -> None:
    deadline = time.monotonic() + timeout
    with httpx.Client(base_url=BASE_URL) as client:
        while time.monotonic() < deadline:
            try:
                if client.get("/health").status_code == 200:
                    return
            except httpx.ConnectError:
                time.sleep(0.1)
    raise RuntimeError("Server did not start within the timeout.")


load_dotenv()
server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
_wait_for_server()
print("Server is up.")

## GET /health

In [ ]:
with httpx.Client(base_url=BASE_URL, timeout=30.0) as client:
    health = client.get("/health")
print(health.status_code, health.json())

## POST /search — Retrieval Only, No LLM Call

In [ ]:
query = "did the company beat earnings expectations?"

with httpx.Client(base_url=BASE_URL, timeout=30.0) as client:
    search = client.post("/search", json={"query": query, "n_results": 3})

print(search.status_code)
for result in search.json():
    print(f"[{result['distance']:.3f}] ({result['ticker']}) {result['text']}")

## POST /search — With a Ticker Filter

In [ ]:
with httpx.Client(base_url=BASE_URL, timeout=30.0) as client:
    filtered = client.post("/search", json={"query": query, "n_results": 5, "ticker": "AAPL"})

print(filtered.status_code)
for result in filtered.json():
    print(f"[{result['distance']:.3f}] ({result['ticker']}) {result['text']}")

## POST /ask — Retrieval + Generation

In [ ]:
with httpx.Client(base_url=BASE_URL, timeout=30.0) as client:
    ask = client.post("/ask", json={"query": query, "n_results": 3})

print(ask.status_code)
if ask.status_code == 200:
    body = ask.json()
    print("answer:", body["answer"])
    print("citations:", body["citations"])
    for source in body["sources"]:
        print(f"  [{source['distance']:.3f}] ({source['ticker']}) {source['text']}")
else:
    # A 500 here usually means LLM_API_KEY/LLM_MODEL aren't set.
    print(ask.text or "(no response body — check LLM_API_KEY/LLM_MODEL are set)")